In [1]:
import pandas as pd
import numpy as np

In [2]:
final_df=pd.read_csv(r"D:\master table.csv")

In [3]:
final_df.shape

(520, 215)

In [7]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 520 entries, 0 to 519
Columns: 215 entries, Transaction_ID to Status.4
dtypes: float64(26), int64(23), object(166)
memory usage: 873.6+ KB


# ___________________Financial Features _____________________

# Feature 1 :  Outstanding Amount

In [13]:
final_df["Outstanding_Amount"]=final_df["Net_Billed_Amount"]- final_df["Total_Collections_Posted"]

In [15]:
final_df["Outstanding_Amount"]

0        704.68
1        411.71
2        920.73
3        938.45
4        478.55
         ...   
515    10573.69
516     4611.82
517      680.83
518     1042.61
519      499.93
Name: Outstanding_Amount, Length: 520, dtype: float64

# Feature 2 : Collection Percentage

In [29]:
final_df["Collection_Percentage"] = np.where(final_df["Net_Billed_Amount"] > 0,
                                    (final_df["Total_Collections_Posted"] / final_df["Net_Billed_Amount"]) * 100,0).round(2)

In [31]:
final_df["Collection_Percentage"]

0      95.59
1      97.59
2      96.05
3      94.95
4      94.90
       ...  
515     0.00
516    45.73
517    95.60
518    95.91
519    96.29
Name: Collection_Percentage, Length: 520, dtype: float64

# Feature 3 :  Net Profit

In [35]:
final_df["Net_Profit"] = (final_df["Total_Collections_Posted"]- final_df["Cost_of_Billing_Operations"])

In [37]:
final_df["Net_Profit"] 

0      14988.37
1      16385.86
2      21987.85
3      17326.51
4       8719.16
         ...   
515     -184.58
516     3715.70
517    14527.54
518    24035.07
519    12728.34
Name: Net_Profit, Length: 520, dtype: float64

# Feature 4 : Profit Margin

In [40]:
final_df["Profit_Margin"] = np.where(final_df["Total_Collections_Posted"] > 0,(final_df["Net_Profit"] /
                                                                               final_df["Total_Collections_Posted"]) * 100,0).round(2)

In [42]:
final_df["Profit_Margin"]

0      98.24
1      98.34
2      98.24
3      98.22
4      97.94
       ...  
515     0.00
516    95.60
517    98.14
518    98.32
519    98.11
Name: Profit_Margin, Length: 520, dtype: float64

# Feature 5 : Write-Off Percentage

In [45]:
final_df["Write_Off_Percentage"] = np.where(final_df["Net_Billed_Amount"] > 0,(final_df["Write_Off_Amount"] /
                                                                               final_df["Net_Billed_Amount"]) * 100,0).round(2)

In [47]:
final_df["Write_Off_Percentage"]

0       4.41
1       2.41
2       0.00
3       0.00
4       0.00
       ...  
515     0.00
516    54.27
517     4.40
518     4.09
519     0.00
Name: Write_Off_Percentage, Length: 520, dtype: float64

# Feature 6 : Collection Gap

In [52]:
final_df["Collection_Gap"] = (final_df["Net_Billed_Amount"]- final_df["Net_Realized_Revenue"])

In [54]:
final_df["Collection_Gap"]

0       704.68
1       411.71
2         0.00
3         0.00
4         0.00
        ...   
515       0.00
516    4611.82
517     680.83
518    1042.61
519       0.00
Name: Collection_Gap, Length: 520, dtype: float64

# Validate New Features

In [60]:
final_df[[
        "Outstanding_Amount",
        "Collection_Percentage",
        "Net_Profit",
        "Profit_Margin",
        "Write_Off_Percentage",
        "Collection_Gap"]].head()

,Outstanding_Amount,Collection_Percentage,Net_Profit,Profit_Margin,Write_Off_Percentage,Collection_Gap
0,704.68,95.59,14988.37,98.24,4.41,704.68
1,411.71,97.59,16385.86,98.34,2.41,411.71
2,920.73,96.05,21987.85,98.24,0.00,0.00
3,938.45,94.95,17326.51,98.22,0.00,0.00
4,478.55,94.90,8719.16,97.94,0.00,0.00


# ________________________ Claim Features__________________________________

# Feature 1 : Claim Size

In [68]:
final_df["Claim_Size"] = np.select(

    [   final_df["Net_Billed_Amount"] < 1000,
        final_df["Net_Billed_Amount"].between(1000, 4999),
        final_df["Net_Billed_Amount"].between(5000, 9999),
        final_df["Net_Billed_Amount"] >= 10000   ],

    [   "Small",
        "Medium",
        "Large",
        "Very Large"  ],  default="Unknown" ) 

In [70]:
final_df["Claim_Size"]

0      Very Large
1      Very Large
2      Very Large
3      Very Large
4           Large
          ...    
515    Very Large
516         Large
517    Very Large
518    Very Large
519    Very Large
Name: Claim_Size, Length: 520, dtype: object

# Feature 2 : High Value Claim

In [73]:
final_df["High_Value_Claim"] = np.where(final_df["High_Value_Invoice_Flag"] == "Yes","Yes","No")

In [75]:
final_df["High_Value_Claim"]

0      No
1      No
2      No
3      No
4      No
       ..
515    No
516    No
517    No
518    No
519    No
Name: High_Value_Claim, Length: 520, dtype: object

# Feature 3 : Approval Flag

In [78]:
final_df["Approval_Flag"] = np.select(

  [  final_df["Adjudication_Decision_Status"] == "Fully Approved",
     final_df["Adjudication_Decision_Status"] == "Conditionally Approved",
     final_df["Adjudication_Decision_Status"].isin(["Rejected at Clearinghouse","Payer Information Request"])],

    [ "Approved","Pending","Rejected" ], 
       default="Unknown")

In [80]:
final_df["Approval_Flag"]

0      Rejected
1      Approved
2      Approved
3      Approved
4      Approved
         ...   
515     Pending
516    Approved
517    Rejected
518    Approved
519    Approved
Name: Approval_Flag, Length: 520, dtype: object

# Feature 4 : Denial Flag

In [83]:
final_df["Denial_Flag"] = np.where(final_df["Adjudication_Decision_Status"] == "Rejected at Clearinghouse", "Yes", "No")

In [85]:
final_df["Denial_Flag"]

0       No
1       No
2       No
3       No
4       No
      ... 
515     No
516     No
517    Yes
518     No
519     No
Name: Denial_Flag, Length: 520, dtype: object

# Feature 5 : Submission Risk

In [92]:
final_df["Submission_Risk"] = np.select(

    [   final_df["Submission_Attempts_Count"] == 1,
        final_df["Submission_Attempts_Count"].between(2,3),
        final_df["Submission_Attempts_Count"] >= 4],

    [   "Low",

        "Medium",

        "High"   ], default="Unknown")

In [96]:
final_df["Submission_Risk"]

0         Low
1         Low
2         Low
3         Low
4         Low
        ...  
515       Low
516    Medium
517       Low
518       Low
519       Low
Name: Submission_Risk, Length: 520, dtype: object

# Validate New Features

In [101]:
final_df[[
        "Claim_Size",
        "High_Value_Claim",
        "Approval_Flag",
        "Denial_Flag",
        "Submission_Risk",]].head()

,Claim_Size,High_Value_Claim,Approval_Flag,Denial_Flag,Submission_Risk
0,Very Large,No,Rejected,No,Low
1,Very Large,No,Approved,No,Low
2,Very Large,No,Approved,No,Low
3,Very Large,No,Approved,No,Low
4,Large,No,Approved,No,Low


# ________________________ Collection & AR Feature ______________________________
# _____________ Accounts Receivable aur payment performance _______

# Feature 1 : Payment Status

In [108]:
final_df["Payment_Status"] = np.select(

    [   final_df["Outstanding_AR_Balance"] == 0,

        (final_df["Outstanding_AR_Balance"] > 0) & (final_df["Total_Collections_Posted"] > 0),

        final_df["Total_Collections_Posted"] == 0],

    [   "Paid",
        "Partial",
        "Pending"   ], default="Unknown" )

In [110]:
final_df["Payment_Status"]

0         Paid
1         Paid
2      Partial
3      Partial
4      Partial
        ...   
515    Pending
516       Paid
517       Paid
518       Paid
519    Partial
Name: Payment_Status, Length: 520, dtype: object

# Feature 2 : Payment Delay Bucket

In [113]:
final_df["Payment_Delay_Bucket"] = np.select(

    [   final_df["Days_to_Payment_Settlement"] <= 30,

        final_df["Days_to_Payment_Settlement"].between(31,60),

        final_df["Days_to_Payment_Settlement"].between(61,90),

        final_df["Days_to_Payment_Settlement"] > 90   ],

    [   "0-30 Days",
        "31-60 Days",
        "61-90 Days",
        "90+ Days"  ], default="Unknown" )

In [117]:
final_df["Payment_Delay_Bucket"]

0      31-60 Days
1      31-60 Days
2        90+ Days
3        90+ Days
4        90+ Days
          ...    
515      90+ Days
516     0-30 Days
517     0-30 Days
518     0-30 Days
519    31-60 Days
Name: Payment_Delay_Bucket, Length: 520, dtype: object

# Feature 3 : Collection Efficiency

In [121]:
final_df["Collection_Efficiency"] = np.select(

    [   final_df["Collection_Percentage"] >= 90,

        final_df["Collection_Percentage"].between(70,89.99),

        final_df["Collection_Percentage"] < 70 ],

    [   "Excellent",
        "Average",
        "Poor"  ],  default="Unknown" )

In [123]:
final_df["Collection_Efficiency"]

0      Excellent
1      Excellent
2      Excellent
3      Excellent
4      Excellent
         ...    
515         Poor
516         Poor
517    Excellent
518    Excellent
519    Excellent
Name: Collection_Efficiency, Length: 520, dtype: object

# Feature 4 : AR Risk

In [162]:
final_df["AR_Risk"] = np.select([final_df["Payment_Delay_Bucket"].isin(["0-30 Days","31-60 Days"]),
                                 final_df["Payment_Delay_Bucket"] == "61-90 Days",
                                 final_df["Payment_Delay_Bucket"] == "90+ Days"],

    [   "Low",
        "Medium",
        "High"   ], default="Unknown")

In [164]:
final_df["AR_Risk"]

0       Low
1       Low
2      High
3      High
4      High
       ... 
515    High
516     Low
517     Low
518     Low
519     Low
Name: AR_Risk, Length: 520, dtype: object

# Feature 5 : Settlement Speed

In [179]:
final_df["Settlement_Speed"] = np.select([final_df["Days_to_Payment_Settlement"] <= 30,
                                          final_df["Days_to_Payment_Settlement"].between(31,60),
                                          final_df["Days_to_Payment_Settlement"] > 60],
                                         
                                         [  "Fast",
                                            "Normal",
                                            "Slow"    ],  default="Unknown")

In [181]:
final_df["Settlement_Speed"] 

0      Normal
1      Normal
2        Slow
3        Slow
4        Slow
        ...  
515      Slow
516      Fast
517      Fast
518      Fast
519    Normal
Name: Settlement_Speed, Length: 520, dtype: object

# Validate New Features

In [190]:
final_df[[
        "Payment_Status",
        "Payment_Delay_Bucket",
        "Settlement_Speed",
        "Collection_Efficiency",
        "AR_Risk",]].head()

,Payment_Status,Payment_Delay_Bucket,Settlement_Speed,Collection_Efficiency,AR_Risk
0,Paid,31-60 Days,Normal,Excellent,Low
1,Paid,31-60 Days,Normal,Excellent,Low
2,Partial,90+ Days,Slow,Excellent,High
3,Partial,90+ Days,Slow,Excellent,High
4,Partial,90+ Days,Slow,Excellent,High


# _________________________  Patient Features ____________________________________

# Feature 1 : Age Group

In [198]:
final_df["Age_Group"] = np.select([final_df["Age"] < 18, 
                                   final_df["Age"].between(18,59),
                                   final_df["Age"] >= 60],
    [  "Child",
        "Adult",
        "Senior"],default="Unknown")

In [200]:
final_df["Age_Group"]

0      Adult
1      Child
2      Adult
3      Adult
4      Adult
       ...  
515    Adult
516    Adult
517    Adult
518    Adult
519    Adult
Name: Age_Group, Length: 520, dtype: object

# Feature 2 : Chronic Patient

In [205]:
final_df["Chronic_Patient"] = np.where(final_df["Chronic_Disease_Flag"] == "Yes","Yes","No")

In [207]:
final_df["Chronic_Patient"]

0      Yes
1       No
2       No
3       No
4       No
      ... 
515     No
516     No
517     No
518     No
519    Yes
Name: Chronic_Patient, Length: 520, dtype: object

# _______________________ Insurance Feature __________________________________

# Feature 1 : Coverage Category

In [220]:
final_df["Coverage_Category"] = np.select(
    [
        final_df["Coverage_Percentage"] < 60,
        final_df["Coverage_Percentage"].between(60, 89.99),
        final_df["Coverage_Percentage"] >= 90
    ],

    [
        "Low",
        "Medium",
        "High"
    ],

    default="Unknown")

In [223]:
final_df["Coverage_Category"] 

0      Medium
1        High
2        High
3      Medium
4      Medium
        ...  
515    Medium
516      High
517      High
518      High
519    Medium
Name: Coverage_Category, Length: 520, dtype: object

# Feature 2 : Insurance Risk

In [236]:
final_df["Insurance_Risk"] = np.select(

    [
        final_df["Coverage_Percentage"] >= 90,
        final_df["Coverage_Percentage"].between(60, 89.99),
        final_df["Coverage_Percentage"] < 60
    ],

    [
        "Low",
        "Medium",
        "High"
    ],default="Unknown")

In [238]:
final_df["Insurance_Risk"]

0      Medium
1         Low
2         Low
3      Medium
4      Medium
        ...  
515    Medium
516       Low
517       Low
518       Low
519    Medium
Name: Insurance_Risk, Length: 520, dtype: object

# _________________________ Operational Feature ______________________________

# Feature 1 : Processing Efficiency

In [266]:
final_df["Processing_Efficiency"] = np.select(

    [
        final_df["Resource_Efficiency_Score"] >= 4.0,

        final_df["Resource_Efficiency_Score"].between(3.00,3.99),

        final_df["Resource_Efficiency_Score"] < 3.00
    ],

    [
        "Excellent",
        "Average",
        "Poor"
    ],

    default="Unknown")

In [268]:
final_df["Processing_Efficiency"]

0        Average
1      Excellent
2      Excellent
3      Excellent
4      Excellent
         ...    
515      Average
516    Excellent
517      Average
518    Excellent
519    Excellent
Name: Processing_Efficiency, Length: 520, dtype: object

# Feature 2 : Shift Performance

In [293]:
final_df["Shift_Performance"] = np.select(

    [

        final_df["Productivity_Index"] >= 85,

        final_df["Productivity_Index"].between(60,84.99),

        final_df["Productivity_Index"] < 60],

    [

        "High",

        "Average",

        "Low"], default="Unknown")

In [295]:
final_df["Shift_Performance"]

0         High
1      Average
2      Average
3         High
4      Average
        ...   
515    Average
516       High
517    Average
518    Average
519       High
Name: Shift_Performance, Length: 520, dtype: object

# _________________________  Business Features  ____________________________

# Feature 1 : Overall Bussiness Health Status

In [301]:
final_df["Overall_Health_Status"] = np.select(

    [   (final_df["Collection_Percentage"] >= 90) &
        (final_df["Profit_Margin"] >= 20),

        (final_df["Collection_Percentage"] >= 75) &
        (final_df["Profit_Margin"] >= 10),

        (final_df["Collection_Percentage"] >= 60),

        (final_df["Collection_Percentage"] < 60) ],

    [  "Excellent",

        "Good",

        "Average",

        "Poor" ],  default="Unknown" )

In [303]:
final_df["Overall_Health_Status"]

0      Excellent
1      Excellent
2      Excellent
3      Excellent
4      Excellent
         ...    
515         Poor
516         Poor
517    Excellent
518    Excellent
519    Excellent
Name: Overall_Health_Status, Length: 520, dtype: object

In [307]:
final_df.shape

(520, 238)

In [309]:
final_df.to_csv("Master_df.csv",index=False)

print("master_df Saved Successfully.")

master_df Saved Successfully.
